# MTI850 – Projet final A26 : Livrable partiel 1
**Demande et rééquilibrage du réseau BIXI à Montréal** — Fortin Laurine, Beaugeois Carole, Elfassi Ethan

### 1. Présentation du problème et description des données
- **Problème :** Prédire la demande horaire (départs/arrivées) par station BIXI pour optimiser le rééquilibrage des vélos.
- **Pertinence Big Data :** >30–40M de trajets historiques (Go), jointures multi-sources (trajets, stations, météo, calendrier).
- **Sources :** [BIXI Open Data](https://bixi.com/en/open-data/) | [Portail Montréal](https://donnees.montreal.ca/group/transport?organization=bixi) | **URL téléchargement :** `<TODO: Ajouter lien Google Drive / OneDrive>`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, VectorAssembler
import matplotlib.pyplot as plt
import seaborn as sns

# Chemins des données (à adapter selon l'environnement local ou HDFS)
TRIPS_PATH = "data/bixi_trips_*.csv"
WEATHER_PATH = "data/montreal_weather.csv"  # optionnel
HDFS_RAW_PATH = "hdfs:///user/mti850/bixi/raw_trips"
OUTPUT_PARQUET_PATH = "df_features.parquet"
HDFS_OUTPUT_PARQUET_PATH = "hdfs:///user/mti850/bixi/df_features.parquet"

### 2. Ingestion de données

In [ ]:
spark = SparkSession.builder \
    .appName("MTI850 - BIXI Montreal - Livrable 1") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

In [ ]:
df_raw = spark.read.csv(TRIPS_PATH, header=True, inferSchema=True)

df_raw.printSchema()
df_raw.show(5, truncate=False)
print(f"Nombre total de lignes brutes : {df_raw.count():,}")

In [ ]:
# Optionnel : sauvegarde brute dans HDFS
# df_raw.write.mode("overwrite").parquet(HDFS_RAW_PATH)

### 3. Nettoyage et prétraitement des données

In [ ]:
# Valeurs manquantes par colonne
null_counts = df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_raw.columns
])
null_counts.show(vertical=True)

# Suppression des doublons et des trajets sans station ou horodatage
df_clean = df_raw.dropDuplicates().dropna(subset=[
    "STARTSTATIONNAME", "ENDSTATIONNAME", "STARTTIMEMS", "ENDTIMEMS"
])

In [ ]:
# Conversion des timestamps (millisecondes -> timestamp) et calcul de la durée en secondes
df_clean = df_clean \
    .withColumn("start_ts", (F.col("STARTTIMEMS").cast("double") / 1000.0).cast("timestamp")) \
    .withColumn("end_ts", (F.col("ENDTIMEMS").cast("double") / 1000.0).cast("timestamp")) \
    .withColumn("duration_sec", F.col("end_ts").cast("long") - F.col("start_ts").cast("long"))

# Normalisation des textes (trim)
df_clean = df_clean \
    .withColumn("STARTSTATIONNAME", F.trim(F.col("STARTSTATIONNAME"))) \
    .withColumn("ENDSTATIONNAME", F.trim(F.col("ENDSTATIONNAME"))) \
    .withColumn("STARTSTATIONARRONDISSEMENT", F.trim(F.col("STARTSTATIONARRONDISSEMENT")))

# Filtrage des valeurs aberrantes (durée entre 60s et 24h, coordonnées valides autour de Montréal)
df_clean = df_clean.filter(
    (F.col("duration_sec").between(60, 86400)) &
    (F.col("STARTSTATIONLATITUDE").between(45.3, 45.8)) &
    (F.col("STARTSTATIONLONGITUDE").between(-74.0, -73.3))
)

# Imputation des arrondissements manquants
df_clean = df_clean.na.fill({"STARTSTATIONARRONDISSEMENT": "Inconnu", "ENDSTATIONARRONDISSEMENT": "Inconnu"})

print(f"Lignes après nettoyage : {df_clean.count():,}")
df_clean.select("STARTSTATIONNAME", "ENDSTATIONNAME", "start_ts", "end_ts", "duration_sec").show(5, truncate=False)

### 4. Analyse exploratoire des données (EDA)

In [ ]:
df_eda = df_clean \
    .withColumn("hour", F.hour("start_ts")) \
    .withColumn("day_of_week", F.dayofweek("start_ts")) \
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin(1, 7), "Fin de semaine").otherwise("Semaine"))

# 1. Demande horaire (Semaine vs Fin de semaine)
hourly_pd = df_eda.groupBy("hour", "is_weekend").count().orderBy("hour").toPandas()

plt.figure(figsize=(10, 4))
sns.lineplot(data=hourly_pd, x="hour", y="count", hue="is_weekend", marker="o")
plt.title("Volume horaire des trajets BIXI")
plt.xlabel("Heure de la journée")
plt.ylabel("Nombre de départs")
plt.xticks(range(0, 24))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 2. Top 10 des stations de départ
top_stations_pd = df_eda.groupBy("STARTSTATIONNAME").count() \
    .orderBy(F.desc("count")).limit(10).toPandas()

plt.figure(figsize=(10, 4))
sns.barplot(data=top_stations_pd, y="STARTSTATIONNAME", x="count", palette="viridis")
plt.title("Top 10 des stations de départ")
plt.xlabel("Nombre de départs")
plt.ylabel("Station")
plt.tight_layout()
plt.show()

In [ ]:
# 3. Distribution de la durée des trajets (échantillon)
sample_pd = df_eda.select("duration_sec", "hour", "day_of_week", "STARTSTATIONLATITUDE", "STARTSTATIONLONGITUDE") \
    .sample(withReplacement=False, fraction=0.01, seed=42).limit(5000).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(sample_pd[sample_pd["duration_sec"] <= 3600]["duration_sec"] / 60, bins=40, kde=True, ax=axes[0])
axes[0].set_title("Distribution de la durée des trajets (<= 60 min)")
axes[0].set_xlabel("Durée (minutes)")

sns.heatmap(sample_pd.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1])
axes[1].set_title("Matrice de corrélation (échantillon)")
plt.tight_layout()
plt.show()

### 5. Sélection des caractéristiques et ingénierie

In [ ]:
# Agrégation horaire des départs par station
departures = df_clean \
    .withColumn("hour_ts", F.date_trunc("hour", "start_ts")) \
    .groupBy(
        F.col("STARTSTATIONNAME").alias("station_name"),
        F.col("STARTSTATIONARRONDISSEMENT").alias("arrondissement"),
        "hour_ts"
    ).agg(
        F.count("*").alias("departures"),
        F.avg("duration_sec").alias("avg_duration_sec"),
        F.first("STARTSTATIONLATITUDE").alias("latitude"),
        F.first("STARTSTATIONLONGITUDE").alias("longitude")
    )

# Agrégation horaire des arrivées par station
arrivals = df_clean \
    .withColumn("hour_ts", F.date_trunc("hour", "end_ts")) \
    .groupBy(F.col("ENDSTATIONNAME").alias("station_name"), "hour_ts") \
    .agg(F.count("*").alias("arrivals"))

# Jointure départs + arrivées et calcul du flux net
df_hourly = departures.join(arrivals, on=["station_name", "hour_ts"], how="left") \
    .na.fill({"arrivals": 0}) \
    .withColumn("net_flow", F.col("arrivals") - F.col("departures"))

In [ ]:
# Variables temporelles
df_hourly = df_hourly \
    .withColumn("hour", F.hour("hour_ts")) \
    .withColumn("day_of_week", F.dayofweek("hour_ts")) \
    .withColumn("month", F.month("hour_ts")) \
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin(1, 7), 1).otherwise(0))

# Variables décalées (lag 1h et 24h par station)
w_station = Window.partitionBy("station_name").orderBy("hour_ts")

df_hourly = df_hourly \
    .withColumn("lag_1h_departures", F.lag("departures", 1).over(w_station)) \
    .withColumn("lag_1h_arrivals", F.lag("arrivals", 1).over(w_station)) \
    .withColumn("lag_24h_departures", F.lag("departures", 24).over(w_station)) \
    .na.fill({"lag_1h_departures": 0, "lag_1h_arrivals": 0, "lag_24h_departures": 0})

In [ ]:
# Encodage des variables catégorielles
station_indexer = StringIndexer(inputCol="station_name", outputCol="station_idx", handleInvalid="keep")
arr_indexer = StringIndexer(inputCol="arrondissement", outputCol="arrondissement_idx", handleInvalid="keep")

df_indexed = station_indexer.fit(df_hourly).transform(df_hourly)
df_indexed = arr_indexer.fit(df_indexed).transform(df_indexed)

# Assemblage du vecteur de caractéristiques
feature_cols = [
    "station_idx", "arrondissement_idx",
    "latitude", "longitude",
    "hour", "day_of_week", "month", "is_weekend",
    "avg_duration_sec",
    "lag_1h_departures", "lag_1h_arrivals", "lag_24h_departures"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
df_features = assembler.transform(df_indexed)

df_features.select("station_name", "hour_ts", "departures", "arrivals", "net_flow", "features").show(5, truncate=False)

### 6. Sauvegarder les données préparées

In [ ]:
# Sauvegarde locale en Parquet
df_features.write.mode("overwrite").parquet(OUTPUT_PARQUET_PATH)

# Sauvegarde HDFS (à décommenter sur le cluster)
# df_features.write.mode("overwrite").parquet(HDFS_OUTPUT_PARQUET_PATH)

# Vérification
df_check = spark.read.parquet(OUTPUT_PARQUET_PATH)
print(f"Parquet sauvegardé : {df_check.count():,} lignes, {len(df_check.columns)} colonnes")
df_check.printSchema()